# ML Challenge 2026 — Colab Setup + Dev Sample

**Goal:** get the data on Colab, audit it, and build a small (~20k S1) dev sample
we can iterate on fast — in this sandbox and on Colab.

**Steps:** 1) env check → 2) clone + pull data (~2.35 GB, one-time, ~5–15 min) →
3) install deps → 4) audit → 5) build sample → 6) download sample.

Run cells top to bottom (Shift+Enter each).

In [ ]:
# 1) Environment check: GPU? RAM? disk?
!nvidia-smi -L 2>/dev/null || echo "no GPU in this runtime"
!free -h | head -2
!df -h /content | tail -1
import sys
print(sys.version)

In [ ]:
# 2) Clone the repo branch + pull the LFS dataset (one-time, a few minutes).
# NOTE: dataset is ~2.35 GB. Runtime must have disk space (Colab free: OK).
!apt-get -qq update && apt-get -qq install -y git-lfs > /dev/null 2>&1
!git lfs install
!git clone -b arena/01a0df98-ml-challenge https://github.com/Saadmadni84/ML-challenge.git
%cd ML-challenge
!git lfs pull
!ls -lh dataset/train dataset/test

In [ ]:
# 3) Install Python dependencies.
!pip -q install -r code/business_entity_resolution/requirements.txt
!python3 -c "import pandas, sklearn; print(pandas.__version__, sklearn.__version__)"

In [ ]:
# 4a) Audit Source-1 + ground truth (streams the files, low RAM).
!python3 code/business_entity_resolution/src/audit.py --source dataset/train/train_source1.tsv --examples 2
!python3 code/business_entity_resolution/src/audit.py --ground-truth dataset/train/train_ground_truth.tsv

In [ ]:
# 4b) Row counts for the big files (fast). Full audit of S2/S3 is optional.
!python3 code/business_entity_resolution/src/audit.py --source dataset/train/train_source2.tsv --counts-only
!python3 code/business_entity_resolution/src/audit.py --source dataset/train/train_source3.tsv --counts-only
!python3 code/business_entity_resolution/src/audit.py --source dataset/test/test_source1.tsv --counts-only

In [ ]:
# 5) Build the stratified dev sample (~20k S1 + targets + distractors).
!python3 code/business_entity_resolution/src/sample.py --train-dir dataset/train --out-dir sample --n-s1 20000 --distractors-per-s1 5 --seed 42
!ls -lh sample/

In [ ]:
# 6) Peek at the sample (small enough for pandas now).
import pandas as pd
gt = pd.read_csv("sample/sample_gt.tsv", sep="\t")
print(gt.shape)
print(gt.head(5).to_string())
n_match = gt["matched_entity_ids"].fillna("").apply(lambda s: 0 if s == "" else s.count(",") + 1)
print(n_match.value_counts().sort_index())

In [ ]:
# 7) Pack the sample and download it, then ATTACH it in chat
# so we can build/test the pipeline in the sandbox too.
!tar -czf dev_sample.tar.gz sample/
!ls -lh dev_sample.tar.gz
from google.colab import files
files.download("dev_sample.tar.gz")

## What to send back

1. The outputs of cells 4a / 4b / 5 (copy-paste the text).
2. The `dev_sample.tar.gz` file attached in chat.

With those, we verify the audit numbers, then design blocking + baseline
on the sample and scale the exact same scripts to full data afterward.